# Real-ESRGAN 4K Video Upscaler (Anime Model)

**Before running:**
1. Go to **Runtime → Change runtime type → GPU** (T4 is fine)
2. Upload your 720p MP4 to Google Drive
3. Set the input/output paths in the **Configuration** cell below

In [ ]:
# Uninstall potentially problematic packages (including torch/torchvision if necessary for a clean install)
!pip uninstall -y realesrgan basicsr torchaudio torchvision torch

# Install specific compatible versions of torch and torchvision first
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install the latest compatible basicsr directly from its GitHub repository
!pip install -q git+https://github.com/xinntao/basicsr.git

# Reinstall realesrgan and other dependencies
!pip install -q realesrgan gfpgan opencv-python-headless

Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 135.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 135.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 99.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 79.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 101.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━

In [ ]:
import torch
print(torch.__version__)

2.5.1+cu121


In [ ]:
import realesrgan
import os
import re

# Construct the full path to realesrgan/utils.py
realesrgan_path = os.path.dirname(realesrgan.__file__)
utils_file_path = os.path.join(realesrgan_path, 'utils.py')

print(f"Attempting to patch: {utils_file_path}")

try:
    with open(utils_file_path, 'r') as f:
        content = f.read()

    # Define the pattern to search for and the replacement string
    # This targets the specific line with hardcoded 'cpu' and will also add 'weights_only=True'
    old_line_pattern = r"loadnet = torch.load\(model_path, map_location=torch.device\('cpu'\)\)"
    new_line = "        loadnet = torch.load(model_path, map_location=torch.device('cuda'), weights_only=True)" # Ensure correct indentation

    if re.search(old_line_pattern, content):
        # Perform the replacement
        patched_content = re.sub(old_line_pattern, new_line, content)

        # Write the patched content back to the file
        with open(utils_file_path, 'w') as f:
            f.write(patched_content)

        print("\u2713 Successfully patched realesrgan/utils.py:")
        print("  - Changed hardcoded 'cpu' to 'cuda' for model loading.")
        print("  - Added 'weights_only=True' to suppress FutureWarning and enhance security.")
        print("\n*Please re-run the Real-ESRGAN model loading cell (`cell wmtcWzrCtoY3`) for these changes to take effect.*")
    else:
        print("Warning: The target line for patching (hardcoded 'cpu' map_location) was not found in the file.")
        print("This might mean the file content has changed, or the issue is no longer present.")

except Exception as e:
    print(f"An error occurred during patching: {e}")

Attempting to patch: /usr/local/lib/python3.12/dist-packages/realesrgan/utils.py
✓ Successfully patched realesrgan/utils.py:
  - Changed hardcoded 'cpu' to 'cuda' for model loading.
  - Added 'weights_only=True' to suppress FutureWarning and enhance security.

*Please re-run the Real-ESRGAN model loading cell (`cell wmtcWzrCtoY3`) for these changes to take effect.*


In [ ]:
import importlib
import realesrgan
import basicsr

print("Reloading realesrgan and basicsr modules...")
importlib.reload(realesrgan)
importlib.reload(basicsr)
print("Modules reloaded. Please re-run the Real-ESRGAN model loading cell (wmtcWzrCtoY3) now.")

Reloading realesrgan and basicsr modules...
Modules reloaded. Please re-run the Real-ESRGAN model loading cell (wmtcWzrCtoY3) now.


In [ ]:
if torch.cuda.is_available():
    print("CUDA is available! PyTorch can use the GPU.")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. PyTorch will use the CPU.")
print(f"PyTorch is currently using device: {next(iter(torch.rand(1).device.__str__().split(':')))}")

CUDA is available! PyTorch can use the GPU.
Current CUDA device: 0
Device name: Tesla T4
PyTorch is currently using device: cpu


In [ ]:
# @title Configuration (Define your video paths and settings here)
INPUT_VIDEO = "/content/drive/MyDrive/Gentle Figment/Gentle Figment.mov" # @param {type:"string"}
OUTPUT_VIDEO = "/content/drive/MyDrive/Gentle Figment/Gentle Figment 4k (Video).mp4" # @param {type:"string"}

# Advanced Settings (Optional)
TILE_SIZE = 0  # @param {type:"integer"} If you have out-of-memory errors, try 200-400
OUTPUT_WIDTH = 3840 # @param {type:"integer"} e.g., 3840 for 4K
OUTPUT_HEIGHT = 2160 # @param {type:"integer"} e.g., 2160 for 4K
CRF = 18 # @param {type:"integer"} Constant Rate Factor for video quality (lower is higher quality, but larger file size)

### Optimize Input Video Location (Optional)

To speed up frame extraction, it's often beneficial to copy the input video from Google Drive to the faster local Colab disk (`/content/`). This section will copy your `INPUT_VIDEO` to `/content/temp_input.mov` and update the `INPUT_VIDEO` variable to point to this local copy. You can then continue with the rest of the notebook.

In [ ]:
import shutil
import os

# Define a local path for the video
LOCAL_INPUT_VIDEO = "/content/temp_input.mov"

# Check if the input video is currently on Google Drive
if INPUT_VIDEO.startswith('/content/drive/'):
    print(f"Copying '{INPUT_VIDEO}' to local Colab disk at '{LOCAL_INPUT_VIDEO}'...")
    try:
        shutil.copyfile(INPUT_VIDEO, LOCAL_INPUT_VIDEO)
        # Update INPUT_VIDEO to point to the local copy
        INPUT_VIDEO = LOCAL_INPUT_VIDEO
        print(f"Copy complete. INPUT_VIDEO is now set to: {INPUT_VIDEO}")
    except FileNotFoundError:
        print(f"Error: Original input video not found at {INPUT_VIDEO}. Please check the path in the Configuration cell.")
    except Exception as e:
        print(f"An error occurred during copying: {e}")
else:
    print(f"Input video is already on local disk or not on Drive: {INPUT_VIDEO}")

# Verify the new path
if os.path.exists(INPUT_VIDEO):
    print(f"Local input video exists: {INPUT_VIDEO}")
else:
    print(f"Error: Local input video does NOT exist at: {INPUT_VIDEO}")

Copying '/content/drive/MyDrive/Gentle Figment/Gentle Figment.mov' to local Colab disk at '/content/temp_input.mov'...
Copy complete. INPUT_VIDEO is now set to: /content/temp_input.mov
Local input video exists: /content/temp_input.mov


In [ ]:
import subprocess
import json
import time # Import time for logging
import os

FRAMES_DIR = "/content/frames_input"
FRAMES_UP_DIR = "/content/frames_upscaled"
os.makedirs(FRAMES_DIR, exist_ok=True);
os.makedirs(FRAMES_UP_DIR, exist_ok=True);

# Get video info
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", INPUT_VIDEO],
    capture_output=True, text=True
)
info = json.loads(probe.stdout)
video_stream = next(s for s in info["streams"] if s["codec_type"] == "video")
fps_parts = video_stream["r_frame_rate"].split("/")
FPS = round(int(fps_parts[0]) / int(fps_parts[1]), 3);
src_w, src_h = int(video_stream["width"]), int(video_stream["height"])
print(f"Source: {src_w}x{src_h} @ {FPS} fps")

# Get total number of frames for better logging
total_frames_expected = None
try:
    # Try to get nb_frames directly from ffprobe
    total_frames_expected = int(video_stream.get('nb_frames', 0))
    if total_frames_expected == 0 and 'format' in info and 'duration' in info['format']:
        # Fallback: estimate from duration and FPS if nb_frames is not present
        duration = float(info['format']['duration'])
        total_frames_expected = int(duration * FPS)
except (KeyError, ValueError):
    pass # Cannot get an accurate estimate

# Extract frames
print("Extracting frames...")
if total_frames_expected:
    print(f"Expecting approximately {total_frames_expected} frames.")

start_extract_time = time.time()
!ffmpeg -y -i "{INPUT_VIDEO}" -qscale:v 2 -threads 0 "{FRAMES_DIR}/frame_%06d.jpg" -loglevel warning
end_extract_time = time.time()

total_frames = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg")])
print(f"\u2713 Extracted {total_frames} frames in {(end_extract_time - start_extract_time):.2f} seconds")

# 5. Upscale Frames with Real-ESRGAN (Anime Model)
# This is the slow step. Progress is printed every 50 frames. If the session disconnects, re-run \u2014 it will skip already-upscaled frames.
import cv2
import numpy as np
# time is already imported above
from realesrgan import RealESRGANer
from basicsr.archs.srvgg_arch import SRVGGNetCompact

# Build the anime video v3 model (uses SRVGGNetCompact, NOT RRDBNet)
model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_conv=16, upscale=4, act_type='prelu')
upsampler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth",
    model=model, # Pass the instantiated model here
    tile=TILE_SIZE,
    tile_pad=10,
    pre_pad=0,
    half=True,  # FP16 for speed on Colab GPUs
    device='cuda' # Explicitly set device to CUDA
)
print("\u2713 Real-ESRGAN anime model loaded")
print(f"Real-ESRGAN model device: {next(upsampler.model.parameters()).device}")

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg"))
already_done = set(os.listdir(FRAMES_UP_DIR))
to_process = [f for f in frames if f not in already_done]
print(f"Frames to upscale: {len(to_process)} ({len(already_done)} already done)")

start = time.time()
for i, fname in enumerate(to_process):
    img = cv2.imread(os.path.join(FRAMES_DIR, fname), cv2.IMREAD_COLOR)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    enhanced, _ = upsampler.enhance(rgb, outscale=4)
    output = cv2.cvtColor(enhanced, cv2.COLOR_RGB2BGR)

    # Resize to exact target resolution
    if output.shape[1] != OUTPUT_WIDTH or output.shape[0] != OUTPUT_HEIGHT:
        output = cv2.resize(output, (OUTPUT_WIDTH, OUTPUT_HEIGHT),
                            interpolation=cv2.INTER_LANCZOS4)

    cv2.imwrite(os.path.join(FRAMES_UP_DIR, fname), output)

    if (i + 1) % 50 == 0 or (i + 1) == len(to_process):
        elapsed = time.time() - start
        fps_rate = (i + 1) / elapsed
        remaining = (len(to_process) - i - 1) / fps_rate
        print(f"  [{i+1}/{len(to_process)}] {fps_rate:.2f} frames/sec \u2014 "
              f"~{remaining/60:.0f} min remaining")

print(f"\n\u2713 Upscaling complete in {(time.time()-start)/60:.1f} minutes")

# 6. Reassemble Upscaled Frames into 4K Video
# Build concat list (handles any gaps gracefully)
up_frames = sorted(f for f in os.listdir(FRAMES_UP_DIR) if f.endswith(".jpg"))
concat_path = "/content/concat_list.txt"
frame_dur = f"{1/FPS:.10f}"
with open(concat_path, "w") as f:
    for fname in up_frames:
        f.write(f"file '{FRAMES_UP_DIR}/{fname}'\n")
        f.write(f"duration {frame_dur}\n")

print(f"Encoding {len(up_frames)} frames \u2192 {OUTPUT_VIDEO}")
print(f"Settings: {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}, {FPS} fps, H.264 CRF {CRF}")

# Encode video with audio from original
!ffmpeg -y \
  -f concat -safe 0 -i "{concat_path}" \
  -i "{INPUT_VIDEO}" \
  -map 0:v:0 -map 1:a:0? \
  -c:v h264_nvenc -crf {CRF} \
  -pix_fmt yuv420p \
  -c:a aac -b:a 320k \
  -r {FPS} \
  -movflags +faststart \
  "{OUTPUT_VIDEO}" \
  -loglevel warning -stats

size_mb = os.path.getsize(OUTPUT_VIDEO) / (1024 * 1024)
print(f"\n\u2713 Done! Saved to: {OUTPUT_VIDEO} ({size_mb:.1f} MB)")

# 7. Cleanup (Optional)
# Delete the temporary frame directories from Colab to free disk space.
import shutil
shutil.rmtree(FRAMES_DIR, ignore_errors=True)
shutil.rmtree(FRAMES_UP_DIR, ignore_errors=True)
if os.path.exists(concat_path):
    os.remove(concat_path)
print("\u2713 Temporary files cleaned up")

Source: 1280x720 @ 29.97 fps
Extracting frames...
Expecting approximately 9960 frames.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x5a3ad8f5c580] stream 0, timescale not set
Guessed Channel Layout for Input Stream #0.0 : stereo
[swscaler @ 0x5a3ad8f97200] deprecated pixel format used, make sure you did set range correctly
✓ Extracted 9960 frames in 167.89 seconds
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth" to /usr/local/lib/python3.12/dist-packages/weights/realesr-animevideov3.pth



100%|██████████| 2.39M/2.39M [00:00<00:00, 56.5MB/s]
/usr/local/lib/python3.12/dist-packages/realesrgan/utils.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loadnet = 

✓ Real-ESRGAN anime model loaded
Real-ESRGAN model device: cuda:0
Frames to upscale: 9960 (0 already done)
  [50/9960] 0.97 frames/sec — ~171 min remaining
  [100/9960] 0.99 frames/sec — ~166 min remaining
  [150/9960] 1.00 frames/sec — ~164 min remaining
  [200/9960] 1.00 frames/sec — ~162 min remaining
  [250/9960] 1.01 frames/sec — ~161 min remaining
  [300/9960] 1.01 frames/sec — ~160 min remaining
  [350/9960] 1.01 frames/sec — ~159 min remaining
  [400/9960] 1.01 frames/sec — ~157 min remaining
  [450/9960] 1.01 frames/sec — ~156 min remaining
  [500/9960] 1.01 frames/sec — ~156 min remaining
  [550/9960] 1.02 frames/sec — ~154 min remaining
  [600/9960] 1.02 frames/sec — ~153 min remaining
  [650/9960] 1.02 frames/sec — ~152 min remaining
  [700/9960] 1.02 frames/sec — ~152 min remaining
  [750/9960] 1.02 frames/sec — ~151 min remaining
  [800/9960] 1.02 frames/sec — ~150 min remaining
  [850/9960] 1.02 frames/sec — ~149 min remaining
  [900/9960] 1.02 frames/sec — ~148 min rema

In [ ]:
print("Monitoring GPU utilization...")
!nvidia-smi